In [ ]:
# WDAN reproduction (She et al., IEEE TNSRE 2023) on BCI IV-2a, 4 classes
# Protocol (paper Sec. II-E): target = Session 2 of A0x (unlabeled during training,
# also the test set); source = Session 1 of the other eight subjects.
import codecs
import os
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

BRANCH = 'feature/wdan-reproduction'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
TARGET_SUBJECTS = list(range(1, 10))
EPOCHS = 200
EXTRA_ARGS = []          # e.g. ['--no_da'], ['--win_sec', '2.0'], ['--zero_phase']
JOB_TAG = 's01_s09'

REPO_PATH = Path(f"/kaggle/working/tcformer-wdan-bcic2a-{JOB_TAG}")
MNE_DATA = Path("/kaggle/working/mne_data")
TRAIN_LOG = Path(f"/kaggle/working/wdan_bcic2a_{JOB_TAG}.log")
RESULT_DIR = Path(f"/kaggle/working/wdan_bcic2a_{JOB_TAG}_results")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )

# Clone the WDAN reproduction branch and install a pinned environment
run([sys.executable, "-m", "pip", "install", "-q", "uv"])

if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, REPO_PATH,
    ])

run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)
WDAN = REPO_PATH / "wdan"

# Fail early if the code does not follow the paper's protocol and preprocessing.
data_text = (WDAN / "data.py").read_text(encoding="utf-8")
train_text = (WDAN / "train.py").read_text(encoding="utf-8")
model_text = (WDAN / "model.py").read_text(encoding="utf-8")
assert 'fmin=8.0, fmax=32.0, order=4' in data_text
assert "exponential_moving_standardize" in data_text
assert 'Xs.append(prep(d["T"][0]))' in data_text
assert 'Xt, yt = data[target]["E"]' in data_text
assert "class CBAM" in model_text and "SUB_BANDS" in model_text
assert 'default=5e-4' in train_text
print("Verified: paper protocol (source = Session 1 of 8 subjects, target/test = Session 2), "
      "8-32 Hz Butterworth + EMS, CBAM + sub-bands + variance layer.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([
    UV, "pip", "install", "--python", PYTHON,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
])
run([UV, "pip", "install", "--python", PYTHON, "-r", "wdan/requirements.txt"], cwd=REPO_PATH)

run(["nvidia-smi"])
run([
    PYTHON, "-c",
    "import torch; print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
    "print('GPU:', torch.cuda.get_device_name(0))",
])

# Train every target subject and export per-subject and per-epoch results
MNE_DATA.mkdir(parents=True, exist_ok=True)

# A rerun in the same draft session must not mix old and new result files.
if RESULT_DIR.exists():
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True)

environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
})

print(
    "===== BCI IV-2a WDAN REPRODUCTION | "
    f"TARGETS={TARGET_SUBJECTS} | EPOCHS={EPOCHS} | EXTRA={EXTRA_ARGS} =====",
    flush=True,
)
print("Live log:", TRAIN_LOG, flush=True)

run([
    PYTHON, "-u", "train.py",
    "--subjects", *TARGET_SUBJECTS,
    "--epochs", EPOCHS,
    "--cache", MNE_DATA / "bci2a_she2023_cache.npz",
    "--out", RESULT_DIR / "results",
    *EXTRA_ARGS,
], cwd=WDAN, env=environment, stream=True, log_path=TRAIN_LOG)

shutil.copy(TRAIN_LOG, RESULT_DIR / TRAIN_LOG.name)
archive = shutil.make_archive(str(RESULT_DIR), "zip", root_dir=RESULT_DIR)
print("Completed targets:", TARGET_SUBJECTS)
print("Kaggle output archive:", archive)
print("Kaggle output log:", TRAIN_LOG)
